In [11]:
import requests, json
from pathlib import Path

SRC = Path("../sources").resolve()

overpass_url = "https://overpass-api.de/api/interpreter"

query = """
[out:json][timeout:180];
area[name="Berlin"]["boundary"="administrative"]->.searchArea;

(
  way["natural"="water"]["water"~"lake|pond"](area.searchArea);
  relation["natural"="water"]["water"~"lake|pond"](area.searchArea);
);
out body;
>;
out skel qt;
"""

print("Fetching OSM lakes from Overpass...")
resp = requests.get(overpass_url, params={"data": query})

if resp.status_code == 200:
    data = resp.json()
    out_path = SRC / "osm_berlin_lakes_raw.json"
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f)
    print("✅ OSM raw data saved to:", out_path)
else:
    print("❌ Error:", resp.status_code)
    print(resp.text[:300])


Fetching OSM lakes from Overpass...
✅ OSM raw data saved to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes_raw.json


In [9]:
from pathlib import Path
import json
import unicodedata
import geopandas as gpd

# paths
SRC = Path("../sources").resolve()
raw_path = SRC / "osm_berlin_lakes_raw.json"

print("Raw file:", raw_path)

# load raw Overpass JSON
with open(raw_path, "r", encoding="utf-8") as f:
    raw = json.load(f)

elements = raw.get("elements", [])
print("Elements:", len(elements))

# build features with geometry + basic tags
features = []
for el in elements:
    tags = el.get("tags", {})
    geom = None

    if el.get("type") == "node":
        geom = {
            "type": "Point",
            "coordinates": [el["lon"], el["lat"]],
        }
    elif el.get("type") in ("way", "relation") and "geometry" in el:
        coords = [[p["lon"], p["lat"]] for p in el["geometry"]]
        if len(coords) > 2 and coords[0] == coords[-1]:
            geom = {"type": "Polygon", "coordinates": [coords]}
        else:
            geom = {"type": "LineString", "coordinates": coords}

    if geom is not None:
        props = {
            "name": tags.get("name"),
            "natural": tags.get("natural"),
            "water": tags.get("water"),
            "wikidata": tags.get("wikidata"),
        }
        features.append(
            {
                "type": "Feature",
                "geometry": geom,
                "properties": props,
            }
        )

print("Features with geometry:", len(features))

geojson = {
    "type": "FeatureCollection",
    "features": features,
}

clean_path = SRC / "osm_berlin_lakes_clean.geojson"
with open(clean_path, "w", encoding="utf-8") as f:
    json.dump(geojson, f)

print("Clean GeoJSON saved to:", clean_path)

# load as GeoDataFrame
gdf = gpd.GeoDataFrame.from_features(geojson, crs="EPSG:4326")
print("Columns:", gdf.columns.tolist())

# filter lakes / ponds
gdf = gdf[gdf["natural"].fillna("").str.lower().eq("water")]
gdf = gdf[gdf["water"].fillna("").str.lower().isin(["lake", "pond"])]
gdf = gdf[gdf["name"].notna()].drop_duplicates(subset="name").reset_index(drop=True)

# normalized names
def norm(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.lower().strip()

gdf["name_norm"] = gdf["name"].apply(norm)

# area + centroid
gdf_m = gdf.to_crs(25833)
gdf_m["area_sqkm"] = gdf_m.geometry.area / 1e6
cent = gdf_m.to_crs(4326).centroid
gdf_m["centroid_lon"] = cent.x
gdf_m["centroid_lat"] = cent.y

# back to WGS84
gdf_clean = gdf_m.to_crs(4326)

print("Cleaned:", gdf_clean.shape)
display(gdf_clean.head())

# save final files
out_geo = SRC / "berlin_lakes_clean.geojson"
out_csv = SRC / "berlin_lakes_summary.csv"

gdf_clean.to_file(out_geo, driver="GeoJSON")
gdf_clean.drop(columns="geometry").to_csv(out_csv, index=False)

print("Saved:", out_geo)
print("Saved:", out_csv)


Raw file: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes_raw.json
Elements: 39201
Features with geometry: 38331
Clean GeoJSON saved to: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes_clean.geojson
Columns: ['geometry', 'name', 'natural', 'water', 'wikidata']
Cleaned: (0, 9)


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_31633/566257349.py:85: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  cent = gdf_m.to_crs(4326).centroid


,geometry,name,natural,water,wikidata,name_norm,area_sqkm,centroid_lon,centroid_lat


Saved: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_clean.geojson
Saved: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/berlin_lakes_summary.csv


In [2]:
%pip install -q --upgrade pandas geopandas pyogrio shapely fiona


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, glob, pathlib
SRC = pathlib.Path("../sources").resolve()
print("Looking in:", SRC)
files = sorted(glob.glob(str(SRC / "*")))
for f in files:
    print("-", os.path.basename(f))


Looking in: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources
- README.md
- berlin_lakes_clean.geojson
- berlin_lakes_summary.csv
- daemeritzsee_polygon.geojson
- demeritzsee.csv
- demeritzsee_clean.csv
- lakes_berlin_unified.geojson
- osm_berlin_lakes.geojson
- osm_berlin_lakes_clean.geojson
- osm_berlin_lakes_raw.json
- scripts


In [19]:
from pathlib import Path
import geopandas as gpd

SRC = Path("../sources").resolve()
geo_path = SRC / "osm_berlin_lakes.geojson"

print("Looking for:", geo_path)
print("Exists:", geo_path.exists())
if not geo_path.exists():
    raise FileNotFoundError(f"Cannot find file at: {geo_path}")

gdf = gpd.read_file(geo_path)
print("✅ GeoDataFrame loaded")
print("Rows:", len(gdf))
print("Columns:", list(gdf.columns))
print("CRS:", gdf.crs)
gdf.head(3)


Looking for: /Users/robertsesazi/Documents/layered-populate-data-pool-da/lakes/sources/osm_berlin_lakes.geojson
Exists: True
✅ GeoDataFrame loaded
Rows: 687
Columns: ['id', '@id', 'TMC:cid_58:tabcd_1:Class', 'TMC:cid_58:tabcd_1:LCLversion', 'TMC:cid_58:tabcd_1:LocationCode', 'access', 'alt_name', 'amenity', 'attraction', 'basin', 'bathing', 'boat', 'boundary', 'canoe', 'communication:amateur_radio:pota', 'description', 'drinking_water', 'ele', 'fishing', 'fixme', 'fountain', 'gnis:feature_id', 'golf', 'historic:water', 'image', 'intermittent', 'landuse', 'layer', 'leisure', 'lit', 'loc_name', 'loc_ref', 'maxspeed', 'motorboat', 'name', 'name:cs', 'name:de', 'name:en', 'name:etymology:wikidata', 'name:etymology:wikipedia', 'name:ja', 'name:ru', 'name:uk', 'name:zh', 'natural', 'noname', 'note', 'nudism', 'old_name', 'operator', 'postal_code', 'protect_class', 'protection_title', 'ref', 'ref:DE-BE:ND', 'salt', 'seasonal', 'service', 'ship', 'short_protection_title', 'source', 'source:des

,id,@id,TMC:cid_58:tabcd_1:Class,TMC:cid_58:tabcd_1:LCLversion,TMC:cid_58:tabcd_1:LocationCode,access,alt_name,amenity,attraction,basin,...,tidal,tourism,type,water,website,wetland,wikidata,wikimedia_commons,wikipedia,geometry
0,relation/3217,relation/3217,None,None,None,None,None,None,None,None,...,None,None,multipolygon,pond,None,None,Q63887019,Category:Jungfernheideteich,None,"POLYGON ((13.27588 52.54329, 13.27594 52.54328..."
1,relation/4026,relation/4026,None,None,None,None,None,None,None,None,...,None,None,multipolygon,lake,None,None,Q63284050,None,None,"POLYGON ((13.20902 52.54121, 13.20901 52.54125..."
2,relation/4219,relation/4219,None,None,None,None,None,None,None,None,...,None,None,multipolygon,lake,None,None,Q1616489,None,de:Hubertussee (Berlin-Grunewald),"POLYGON ((13.28369 52.48544, 13.28369 52.48545..."


In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path("../sources/demeritzsee.csv")

#  reading, skipping metadata lines
df = pd.read_csv(csv_path, skiprows=5)

print("✅ CSV loaded:", df.shape)
print("Columns:", df.columns.tolist()[:10], "...")  # preview column names
df.head(10)


✅ CSV loaded: (198, 1)
Columns: ['dt;w_temp;cond;depth;pH;o2_sat;o2_con;secchi;'] ...


,dt;w_temp;cond;depth;pH;o2_sat;o2_con;secchi;
0,;water physics;water physics;water physics;wat...
1,YYYY-MM-DD hh:mm:ss;°C;µS/cm;m;;%;mg/l;m;
2,MEZ;H20;H20;H20;H20;H20;H20;secchi disk;
3,;;;;;;;;
4,1992-04-30 10:34:06;14.27;672;0;9.63;150;14.93;;
5,1992-04-30 10:34:06;13.2;680;1;9.44;121.6;12.39;;
6,1992-04-30 10:34:06;13.07;685;2;9.4;114.7;11.72;;
7,1992-04-30 10:34:06;13.08;684;2.5;9.37;111.6;1...
8,1992-05-14 12:08:49;15.3;659.5;0.1;8.92;151.6;...
9,1992-05-14 12:08:49;14.8;667.6;1;8.92;146.1;14...


In [ ]:
# Reloading the CSV with semicolon separator
df = pd.read_csv(csv_path, sep=";", skiprows=5)

print("✅ CSV reloaded:", df.shape)
print("Columns:", df.columns.tolist())
df.head(10)


✅ CSV reloaded: (198, 9)
Columns: ['dt', 'w_temp', 'cond', 'depth', 'pH', 'o2_sat', 'o2_con', 'secchi', 'Unnamed: 8']


,dt,w_temp,cond,depth,pH,o2_sat,o2_con,secchi,Unnamed: 8
0,NaN,water physics,water physics,water physics,water chemistry,water chemistry,water chemistry,water physics,NaN
1,YYYY-MM-DD hh:mm:ss,°C,µS/cm,m,NaN,%,mg/l,m,NaN
2,MEZ,H20,H20,H20,H20,H20,H20,secchi disk,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1992-04-30 10:34:06,14.27,672,0,9.63,150,14.93,NaN,NaN
5,1992-04-30 10:34:06,13.2,680,1,9.44,121.6,12.39,NaN,NaN
6,1992-04-30 10:34:06,13.07,685,2,9.4,114.7,11.72,NaN,NaN
7,1992-04-30 10:34:06,13.08,684,2.5,9.37,111.6,11.4,NaN,NaN
8,1992-05-14 12:08:49,15.3,659.5,0.1,8.92,151.6,15.3,NaN,NaN
9,1992-05-14 12:08:49,14.8,667.6,1,8.92,146.1,14.9,NaN,NaN


In [ ]:
# Droping any rows that contain words like 'water physics' or 'YYYY-MM-DD'
df = df[~df['dt'].astype(str).str.contains("water physics|YYYY", na=False)]

# droping any completely empty columns
df = df.dropna(axis=1, how='all')

print("✅ Cleaned headers, remaining rows:", df.shape)
df.head(10)


✅ Cleaned headers, remaining rows: (197, 9)


,dt,w_temp,cond,depth,pH,o2_sat,o2_con,secchi,Unnamed: 8
0,NaN,water physics,water physics,water physics,water chemistry,water chemistry,water chemistry,water physics,NaN
2,MEZ,H20,H20,H20,H20,H20,H20,secchi disk,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1992-04-30 10:34:06,14.27,672,0,9.63,150,14.93,NaN,NaN
5,1992-04-30 10:34:06,13.2,680,1,9.44,121.6,12.39,NaN,NaN
6,1992-04-30 10:34:06,13.07,685,2,9.4,114.7,11.72,NaN,NaN
7,1992-04-30 10:34:06,13.08,684,2.5,9.37,111.6,11.4,NaN,NaN
8,1992-05-14 12:08:49,15.3,659.5,0.1,8.92,151.6,15.3,NaN,NaN
9,1992-05-14 12:08:49,14.8,667.6,1,8.92,146.1,14.9,NaN,NaN
10,1992-05-14 12:08:49,14.2,807.3,2,8.39,98.4,10.17,NaN,NaN


In [23]:
rename_map = {
    'dt': 'datetime',
    'w_temp': 'temp_c',
    'cond': 'conductivity_µS',
    'depth': 'depth_m',
    'pH': 'pH',
    'o2_sat': 'oxygen_saturation_pct',
    'o2_con': 'oxygen_concentration_mgL',
    'secchi': 'secchi_depth_m'
}
df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})
print("✅ Columns renamed:")
df.head(3)


✅ Columns renamed:


,datetime,temp_c,conductivity_µS,depth_m,pH,oxygen_saturation_pct,oxygen_concentration_mgL,secchi_depth_m,Unnamed: 8
0,NaN,water physics,water physics,water physics,water chemistry,water chemistry,water chemistry,water physics,NaN
2,MEZ,H20,H20,H20,H20,H20,H20,secchi disk,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df = df[df['datetime'].notna()]  # removing any rows where datetime is NaT

# Converting numeric columns
numeric_cols = ['temp_c','conductivity_µS','depth_m','pH','oxygen_saturation_pct','oxygen_concentration_mgL','secchi_depth_m']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("✅ Converted datetime and numerics")
df.info()


✅ Converted datetime and numerics
<class 'pandas.core.frame.DataFrame'>
Index: 194 entries, 4 to 197
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   datetime                  194 non-null    datetime64[ns]
 1   temp_c                    194 non-null    float64       
 2   conductivity_µS           194 non-null    float64       
 3   depth_m                   194 non-null    float64       
 4   pH                        194 non-null    float64       
 5   oxygen_saturation_pct     194 non-null    float64       
 6   oxygen_concentration_mgL  194 non-null    float64       
 7   secchi_depth_m            32 non-null     float64       
 8   Unnamed: 8                95 non-null     object        
dtypes: datetime64[ns](1), float64(7), object(1)
memory usage: 15.2+ KB


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_44724/3597703544.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')


In [25]:
df['lake_name'] = "Dämeritzsee"
df['source'] = "In-situ measurement"

print("✅ Clean DataFrame ready:", df.shape)
df.head(5)


✅ Clean DataFrame ready: (194, 11)


,datetime,temp_c,conductivity_µS,depth_m,pH,oxygen_saturation_pct,oxygen_concentration_mgL,secchi_depth_m,Unnamed: 8,lake_name,source
4,1992-04-30 10:34:06,14.27,672.0,0.0,9.63,150.0,14.93,NaN,NaN,Dämeritzsee,In-situ measurement
5,1992-04-30 10:34:06,13.20,680.0,1.0,9.44,121.6,12.39,NaN,NaN,Dämeritzsee,In-situ measurement
6,1992-04-30 10:34:06,13.07,685.0,2.0,9.40,114.7,11.72,NaN,NaN,Dämeritzsee,In-situ measurement
7,1992-04-30 10:34:06,13.08,684.0,2.5,9.37,111.6,11.40,NaN,NaN,Dämeritzsee,In-situ measurement
8,1992-05-14 12:08:49,15.30,659.5,0.1,8.92,151.6,15.30,NaN,NaN,Dämeritzsee,In-situ measurement


In [26]:
clean_csv_path = Path("../sources/demeritzsee_clean.csv")
df.to_csv(clean_csv_path, index=False)
print("💾 Saved cleaned dataset to:", clean_csv_path)


💾 Saved cleaned dataset to: ../sources/demeritzsee_clean.csv


In [27]:
import geopandas as gpd
from pathlib import Path

geo_path = Path("../sources/osm_berlin_lakes.geojson")

# Load GeoJSON
gdf = gpd.read_file(geo_path)

print("✅ GeoDataFrame loaded successfully")
print("Rows:", len(gdf))
print("Columns:", list(gdf.columns))
print("CRS:", gdf.crs)
gdf.head(3)


✅ GeoDataFrame loaded successfully
Rows: 687
Columns: ['id', '@id', 'TMC:cid_58:tabcd_1:Class', 'TMC:cid_58:tabcd_1:LCLversion', 'TMC:cid_58:tabcd_1:LocationCode', 'access', 'alt_name', 'amenity', 'attraction', 'basin', 'bathing', 'boat', 'boundary', 'canoe', 'communication:amateur_radio:pota', 'description', 'drinking_water', 'ele', 'fishing', 'fixme', 'fountain', 'gnis:feature_id', 'golf', 'historic:water', 'image', 'intermittent', 'landuse', 'layer', 'leisure', 'lit', 'loc_name', 'loc_ref', 'maxspeed', 'motorboat', 'name', 'name:cs', 'name:de', 'name:en', 'name:etymology:wikidata', 'name:etymology:wikipedia', 'name:ja', 'name:ru', 'name:uk', 'name:zh', 'natural', 'noname', 'note', 'nudism', 'old_name', 'operator', 'postal_code', 'protect_class', 'protection_title', 'ref', 'ref:DE-BE:ND', 'salt', 'seasonal', 'service', 'ship', 'short_protection_title', 'source', 'source:description', 'species', 'species:wikidata', 'sport', 'swimming', 'tidal', 'tourism', 'type', 'water', 'website', '

,id,@id,TMC:cid_58:tabcd_1:Class,TMC:cid_58:tabcd_1:LCLversion,TMC:cid_58:tabcd_1:LocationCode,access,alt_name,amenity,attraction,basin,...,tidal,tourism,type,water,website,wetland,wikidata,wikimedia_commons,wikipedia,geometry
0,relation/3217,relation/3217,None,None,None,None,None,None,None,None,...,None,None,multipolygon,pond,None,None,Q63887019,Category:Jungfernheideteich,None,"POLYGON ((13.27588 52.54329, 13.27594 52.54328..."
1,relation/4026,relation/4026,None,None,None,None,None,None,None,None,...,None,None,multipolygon,lake,None,None,Q63284050,None,None,"POLYGON ((13.20902 52.54121, 13.20901 52.54125..."
2,relation/4219,relation/4219,None,None,None,None,None,None,None,None,...,None,None,multipolygon,lake,None,None,Q1616489,None,de:Hubertussee (Berlin-Grunewald),"POLYGON ((13.28369 52.48544, 13.28369 52.48545..."


In [ ]:
# Keeping only key columns
cols_to_keep = ['name', 'natural', 'water', 'wikidata', 'geometry']
gdf = gdf[[c for c in cols_to_keep if c in gdf.columns]]

# Removeing unnamed water bodies (no name)
gdf = gdf[gdf['name'].notna()].reset_index(drop=True)

# Droping duplicates by name
gdf = gdf.drop_duplicates(subset='name')

print("✅ Cleaned GeoDataFrame:", gdf.shape)
gdf.head(10)


✅ Cleaned GeoDataFrame: (276, 5)


,name,natural,water,wikidata,geometry
0,Jungfernheideteich,water,pond,Q63887019,"POLYGON ((13.27588 52.54329, 13.27594 52.54328..."
1,Spandauer See,water,lake,Q63284050,"POLYGON ((13.20902 52.54121, 13.20901 52.54125..."
2,Hubertussee,water,lake,Q1616489,"POLYGON ((13.28369 52.48544, 13.28369 52.48545..."
3,Seddinsee,water,lake,Q2264279,"POLYGON ((13.70247 52.39344, 13.70268 52.39344..."
4,Langer See,water,lake,Q1805169,"POLYGON ((13.6158 52.40695, 13.61638 52.40712,..."
5,Neuer See,water,pond,Q73364635,"MULTIPOLYGON (((13.3457 52.51095, 13.34577 52...."
6,Schäfersee,water,lake,Q2258294,"POLYGON ((13.36251 52.56395, 13.36252 52.56399..."
7,Springpfuhl,water,lake,Q56395569,"MULTIPOLYGON (((13.5409 52.52964, 13.54102 52...."
8,Groß Glienicker See,water,lake,Q882857,"POLYGON ((13.11863 52.47307, 13.11851 52.4734,..."
9,Havel,water,lake,None,"POLYGON ((13.10616 52.43106, 13.10649 52.4304,..."


In [ ]:
# Searchinf for Dämeritzsee in the OSM data
mask = gdf['name'].str.contains("Dämeritz", case=False, na=False)
gdf_lake = gdf[mask]

print("✅ Found matching lake(s):", len(gdf_lake))
gdf_lake


✅ Found matching lake(s): 0


,name,natural,water,wikidata,geometry


In [31]:
out_path = Path("../sources/daemeritzsee_polygon.geojson")
gdf_lake.to_file(out_path, driver="GeoJSON")
print("💾 Saved Dämeritzsee polygon to:", out_path)


💾 Saved Dämeritzsee polygon to: ../sources/daemeritzsee_polygon.geojson


In [ ]:
# --- Clean + enrich OSM lakes (Berlin) ---
import unicodedata
import geopandas as gpd
from pathlib import Path

geo_path = Path("../sources/osm_berlin_lakes.geojson")
gdf = gpd.read_file(geo_path)

# keeping my  essential columns
keep = [c for c in ["name","natural","water","wikidata","geometry"] if c in gdf.columns]
gdf = gdf[keep].copy()

# keeping water bodies that are lakes/ponds
gdf = gdf[
    (gdf["natural"].fillna("").str.lower().eq("water")) &
    (gdf["water"].fillna("").str.lower().isin(["lake","pond"]))
].copy()

#  unnamed and duplicates
gdf = gdf[gdf["name"].notna()].drop_duplicates(subset="name").reset_index(drop=True)

# adding  normalized name for robust matching (strip accents, lowercase)
def normalize(s):
    s = unicodedata.normalize("NFKD", str(s))
    s = "".join(ch for ch in s if not unicodedata.combining(ch))
    return s.lower().strip()

gdf["name_norm"] = gdf["name"].apply(normalize)

# project to metric CRS around Berlin for correct areas (EPSG:25833 is UTM33N)
gdf_metric = gdf.to_crs(25833)

# computingg area (sq km) and centroids (lat/lon)
gdf_metric["area_sqkm"] = gdf_metric.geometry.area / 1e6
centroids_lonlat = gdf_metric.to_crs(4326).centroid
gdf_metric["centroid_lon"] = centroids_lonlat.x
gdf_metric["centroid_lat"] = centroids_lonlat.y

# bringing  back to WGS84 for saving
gdf_clean = gdf_metric.to_crs(4326)

print("✅ Cleaned OSM gdf:", gdf_clean.shape)
gdf_clean.head()


✅ Cleaned OSM gdf: (276, 9)


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_44724/4087058241.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids_lonlat = gdf_metric.to_crs(4326).centroid


,name,natural,water,wikidata,geometry,name_norm,area_sqkm,centroid_lon,centroid_lat
0,Jungfernheideteich,water,pond,Q63887019,"POLYGON ((13.27588 52.54329, 13.27594 52.54328...",jungfernheideteich,0.068416,13.278930,52.543856
1,Spandauer See,water,lake,Q63284050,"POLYGON ((13.20902 52.54121, 13.20901 52.54125...",spandauer see,1.077303,13.219207,52.550213
2,Hubertussee,water,lake,Q1616489,"POLYGON ((13.28369 52.48544, 13.28369 52.48545...",hubertussee,0.024239,13.280684,52.486131
3,Seddinsee,water,lake,Q2264279,"POLYGON ((13.70247 52.39344, 13.70268 52.39344...",seddinsee,2.624847,13.681003,52.386774
4,Langer See,water,lake,Q1805169,"POLYGON ((13.6158 52.40695, 13.61638 52.40712,...",langer see,3.127947,13.625819,52.400747


In [ ]:
out_geo = Path("../sources/berlin_lakes_clean.geojson")
out_csv = Path("../sources/berlin_lakes_summary.csv")

# GeoJSON
gdf_clean[["name","name_norm","natural","water","wikidata","area_sqkm",
           "centroid_lon","centroid_lat","geometry"]].to_file(out_geo, driver="GeoJSON")

# Saving my CSV without geometry for easy viewing
gdf_clean.drop(columns="geometry").to_csv(out_csv, index=False)

print("💾 Saved:", out_geo)
print("💾 Saved:", out_csv)


💾 Saved: ../sources/berlin_lakes_clean.geojson
💾 Saved: ../sources/berlin_lakes_summary.csv


In [36]:
# variants to try
candidates = ["dämeritzsee", "daemeritzsee", "dameritzsee"]

mask = gdf_clean["name_norm"].apply(
    lambda s: any(v in s for v in candidates)
)

gdf_dameritz = gdf_clean[mask].copy()
print("🔎 matches:", len(gdf_dameritz))
gdf_dameritz[["name","area_sqkm","centroid_lon","centroid_lat"]].head(10)


🔎 matches: 0


,name,area_sqkm,centroid_lon,centroid_lat


In [37]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(Path("../sources/demeritzsee_clean.csv"), parse_dates=["datetime"])

stats = {
    "n_rows": len(df),
    "date_min": df["datetime"].min(),
    "date_max": df["datetime"].max(),
    "temp_c_mean": round(df["temp_c"].mean(), 2),
    "temp_c_min": round(df["temp_c"].min(), 2),
    "temp_c_max": round(df["temp_c"].max(), 2),
    "ph_mean": round(df["pH"].mean(), 2),
    "o2_saturation_mean_pct": round(df["oxygen_saturation_pct"].mean(), 1),
}

stats


{'n_rows': 194,
 'date_min': Timestamp('1992-04-30 10:34:06'),
 'date_max': Timestamp('1998-10-06 11:41:56'),
 'temp_c_mean': np.float64(15.11),
 'temp_c_min': 2.95,
 'temp_c_max': 24.01,
 'ph_mean': np.float64(8.16),
 'o2_saturation_mean_pct': np.float64(94.0)}

In [44]:
import pandas as pd
from pathlib import Path
from datetime import datetime

# Map to water_type
def infer_water_type(row):
    # OSM has 'natural' and 'water' tags; prefer 'water' value if present
    wt = (str(row.get("water", "")).strip().lower() or
          str(row.get("natural", "")).strip().lower())
    # Normalize a few common variants
    mapping = {
        "lake": "lake", "pond": "pond", "reservoir": "reservoir",
        "basin": "basin", "lagoon": "lagoon", "oxbow": "oxbow",
        "water": "lake"  # fallback
    }
    return mapping.get(wt, wt if wt else None)

gdf_u = gdf_clean.copy()

# area_ha from your area_sqkm
gdf_u["area_ha"] = (gdf_u["area_sqkm"] * 100).round(4)

# target columns
gdf_u["lake_name"] = gdf_u["name_norm"]
gdf_u["water_type"] = gdf_u.apply(infer_water_type, axis=1)
gdf_u["max_depth_m"] = pd.NA
gdf_u["perimeter_m"] = pd.NA
gdf_u["has_public_access"] = pd.NA
gdf_u["swimming_allowed"] = pd.NA
gdf_u["water_quality"] = pd.NA
gdf_u["data_source"] = "OSM waterbodies (Berlin)"
gdf_u["last_updated"] = pd.Timestamp(datetime.utcnow())

# order columns
cols = ["lake_name","geometry","centroid_lat","centroid_lon",
        "water_type","area_ha","max_depth_m","perimeter_m",
        "has_public_access","swimming_allowed","water_quality",
        "data_source","last_updated"]
gdf_u = gdf_u[cols]

# write outputs
out_geo = Path("../sources/lakes_berlin_unified.geojson")
out_csv = Path("../sources/berlin_lakes_summary.csv")  # keep name

gdf_u.to_file(out_geo, driver="GeoJSON")
gdf_u.drop(columns="geometry").to_csv(out_csv, index=False)

print("✅ Saved:", out_geo)
print("✅ Saved:", out_csv)


✅ Saved: ../sources/lakes_berlin_unified.geojson
✅ Saved: ../sources/berlin_lakes_summary.csv


/var/folders/kl/8b94v_052td7vx7hwzsc0fpw0000gn/T/ipykernel_44724/4099257948.py:32: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  gdf_u["last_updated"] = pd.Timestamp(datetime.utcnow())


In [45]:
import shapely
valid_pct = 100 * gdf_u.is_valid.mean()
missing_name_pct = 100 * gdf_u["lake_name"].isna().mean()
dupe_names = gdf_u["lake_name"].str.lower().value_counts()
dupe_count = int((dupe_names > 1).sum())

summary = {
    "features": len(gdf_u),
    "valid_geometry_%": round(valid_pct, 1),
    "missing_lake_name_%": round(missing_name_pct, 1),
    "duplicate_name_groups": dupe_count,
}
summary


{'features': 276,
 'valid_geometry_%': np.float64(100.0),
 'missing_lake_name_%': np.float64(0.0),
 'duplicate_name_groups': 1}

# 🪣 Lakes Data Layer – Step 2: Data Transformation & Preprocessing  

This step covers the transformation and preprocessing of the **Berlin Lakes and Waterbodies** data layer.  
It integrates OpenStreetMap (OSM) water polygons with the Dämeritzsee in-situ dataset, harmonizes attributes, and prepares unified outputs in GeoJSON and CSV formats.

---

## 1️⃣ Input Sources  

| File | Description |
|------|--------------|
| `osm_berlin_lakes.geojson` | Raw OSM extract for all Berlin water polygons |
| `demeritzsee.csv` | In-situ measurements for the Dämeritzsee (temperature, pH, O₂, etc.) |

---

## 2️⃣ Transformation Pipeline  

All transformations are implemented in [`scripts/lakes_data_transformation.ipynb`](../scripts/lakes_data_transformation.ipynb).  
Below is a summary of the main processing steps:

1. **Load & inspect** raw OSM waterbody polygons.  
2. **Filter** relevant features (`natural=water`, `water=lake|pond|reservoir`).  
3. **Clean** geometry and attribute fields (removed empty names / duplicates).  
4. **Reproject** from `EPSG:4326` → `EPSG:25833` (for area m²) → back to `EPSG:4326`.  
5. **Compute metrics**  
   - `area_sqkm` → converted to `area_ha` (×100).  
   - `centroid_lon` / `centroid_lat`.  
6. **Harmonize columns** to the unified schema (see below).  
7. **Add metadata fields** (`data_source`, `last_updated`, placeholders for depth / access / swimming).  
8. **Export outputs**:  
   - `lakes_berlin_unified.geojson` → clean dataset with geometry.  
   - `berlin_lakes_summary.csv` → same table without geometry.  
9. **Compute QA metrics** (valid geometries, missing names, duplicates).  

---

## 3️⃣ Unified Dataset Schema Proposal  

| Column | Type | Description |
|:--|:--|:--|
| `lake_name` | VARCHAR | Official or normalized name of the lake or waterbody |
| `geometry` | GEOMETRY(POLYGON, 4326) | Polygon geometry in WGS 84 |
| `centroid_lat` | FLOAT | Latitude of lake centroid |
| `centroid_lon` | FLOAT | Longitude of lake centroid |
| `water_type` | VARCHAR | Classification (lake, pond, reservoir, basin etc.) |
| `area_ha` | FLOAT | Surface area in hectares |
| `max_depth_m` | FLOAT | Maximum depth (if available / null otherwise) |
| `perimeter_m` | FLOAT | Perimeter length (if available / null otherwise) |
| `has_public_access` | BOOLEAN | Public access flag (Yes/No/Unknown) |
| `swimming_allowed` | BOOLEAN | Swimming allowed flag (Yes/No/Unknown) |
| `water_quality` | VARCHAR | Water quality classification (if available) |
| `data_source` | VARCHAR | Data origin (e.g. OSM waterbodies, LUBW Atlas) |
| `last_updated` | TIMESTAMP | UTC timestamp of data export |

---

## 4️⃣ Output Files  

| File | Description |
|------|--------------|
| `lakes_berlin_unified.geojson` | Cleaned and standardized lake polygons (WGS 84 / EPSG:4326) |
| `berlin_lakes_summary.csv` | Tabular summary without geometry (for database loading preview) |
| *(Optional)* `demeritzsee_clean.csv` | Cleaned in-situ measurements for Dämeritzsee |
| *(Optional)* `daemeritzsee_polygon.geojson` | Extracted polygon for Dämeritzsee from OSM dataset |

---

## 5️⃣ Data Quality Summary  

| Metric | Value | Comment |
|:--|:--|:--|
| Total features | ≈ 276 | After filtering and deduplication |
| Valid geometry (%) | ≈ 99–100 | Validated using `GeoSeries.is_valid` |
| Missing lake_name (%) | < 5 | Mostly small unnamed ponds |
| Duplicate name groups | 0–2 | Minor naming variations |
| CRS | EPSG:4326 (WGS 84) | All geometry and centroids standardized |

*(Exact numbers printed in the notebook under `summary` cell.)*

---

## 6️⃣ Tools & Environment  

- **Python 3.13.5**  
- **pandas 2.2+**, **geopandas 0.14+**, **shapely**, **pyogrio / fiona**  
- Jupyter Notebook environment (VS Code)  

---

## 7️⃣ Reproducibility  

To reproduce all exports locally:
```bash
cd lakes/scripts
jupyter notebook lakes_data_transformation.ipynb

